# D1 · Parco fotovoltaico: dalla previsione meteo all'offerta di energia (C#)

Stessi numeri nel notebook [PySharp](D1-fotovoltaico-pysharp.ipynb) (facciata `Interop`); un test verifica che coincidano. Kernel JupyterNet su .NET 11.

## Il caso

Un **parco fotovoltaico da 1 MWp** (inverter da 600 kW AC) vende energia sul mercato del giorno prima. Il SCADA registra ogni ora irraggiamento, temperature, potenza AC e stato dell'inverter; un servizio meteo manda, il giorno prima, la **previsione di irraggiamento** per le 24 ore successive, con il suo errore.

**Chi lavora sui dati.** L'*energy manager* deve presentare l'offerta di energia di domani; chi fa *O&M* vuole sapere se l'impianto rende come dovrebbe (e se vale la pena lavare i moduli); la *proprietà* vuole sapere perché un giorno si è prodotto meno. La domanda del cliente (il trader): **«quanta energia offrite sul mercato domani?»**, e va data **con un'incertezza**, perché produrre meno di quanto offerto costa di più che produrre di più.

**Dati sintetici** (`generate-data.cs`, con seme): 8 settimane orarie. Le prime 7 per addestrare, l'ultima come prova mai vista. La verità nota è in `data/verita.json`: la pulizia dei moduli (giorno 30), un guasto dell'inverter (giorno 38, 5 ore) e l'errore del servizio meteo.

| Passo | Domanda | Strumento |
|---|---|---|
| 1 | L'impianto rende come dovrebbe? | KPI: performance ratio, disponibilità |
| 2 | Quanta potenza darà domani, **sapendo il meteo previsto**? | regressione sulla covariata, seasonal naive, MASE |
| 3 | Con quale incertezza? | quantili dei residui per ora, perdita pinball, copertura |
| 4 | E sull'energia dell'**intera giornata**? | traiettorie campionate (blocchi di residui) |
| 5 | Quanta energia offro? | quantile ottimo dato il costo dell'errore |
| 6 | KPI, dashboard, risposta al cliente | tile, grafici, testo con i numeri |

In [ ]:
#load "../../../shared/MathemaSharp.PlotSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;
using System.IO;
using System.Runtime.CompilerServices;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });
static string Here([CallerFilePath] string path = "") => Path.GetDirectoryName(path)!;   // cartella di questo notebook
static double Num(string s) => double.Parse(s, CultureInfo.InvariantCulture);

var data = Path.GetFullPath(Path.Combine(Here(), "../../../../scenari/D-fotovoltaico/data"));
var rows = File.ReadLines(Path.Combine(data, "impianto_orario.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] G = rows.Select(r => Num(r[1])).ToArray(), Gf = rows.Select(r => Num(r[2])).ToArray(), Tair = rows.Select(r => Num(r[3])).ToArray();
double[] Pac = rows.Select(r => Num(r[5])).ToArray(), ok = rows.Select(r => Num(r[6])).ToArray();
const int trainDays = 49, trainHours = trainDays * 24, total = 56 * 24;
const double pMax = 600, pNom = 1000;   // kW AC massimi dell'inverter, kWp installati

## 1 · Acquisire e misurare: l'impianto rende come dovrebbe?
Il **performance ratio** (IEC 61724) è l'energia prodotta divisa per quella che produrrebbe un impianto ideale con la stessa insolazione: $PR=E_{AC}/(H_{POA}\cdot P_{nom})$. Misura la salute dell'impianto *a prescindere dal meteo*. La **disponibilità** conta le ore di sole in cui l'inverter era in servizio.

In [ ]:
var dailyEnergy = new double[56];
var dailyInsolation = new double[56];
var pr = new double[56];
for (var d = 0; d < 56; d++)
{
    for (var h = 0; h < 24; h++) { dailyEnergy[d] += Pac[d * 24 + h]; dailyInsolation[d] += G[d * 24 + h] / 1000; }
    pr[d] = dailyInsolation[d] > 0.5 ? dailyEnergy[d] / (dailyInsolation[d] * pNom) : double.NaN;
}
double MeanPr(int from, int to) => Enumerable.Range(from, to - from).Select(d => pr[d]).Average();
P("pr_prima_pulizia", MeanPr(24, 30));
P("pr_dopo_pulizia", MeanPr(31, 37));
P("pr_giorno_guasto", pr[38]);
var sunny = Enumerable.Range(0, total).Where(i => G[i] > 50).ToArray();
P("disponibilita", sunny.Count(i => ok[i] == 1) / (double)sunny.Length);
P("ore_al_limite_inverter", Pac.Count(p => p >= pMax - 0.05));

pr_prima_pulizia: 0.779849
pr_dopo_pulizia: 0.808781
pr_giorno_guasto: 0.368400
disponibilita: 0.992636
ore_al_limite_inverter: 17.000000

Il PR sale dopo la pulizia dei moduli (giorno 30) e precipita nel giorno del guasto: il KPI vede entrambi. Ma la differenza dovuta alla pulizia è di pochi punti: serve una **media su più giorni** per distinguerla dal rumore.

In [ ]:
Chart.Lines("Performance ratio giornaliero (pulizia al giorno 30, guasto al 38)", new Line("PR", pr));

Performance ratio giornaliero (pulizia al giorno 30, guasto al 38) 0.3 0.4 0.5 0.6 0.7 0.8 0.9 0 20 40 60 <path d='M64 81.79L74.53 73.98L85.07 80.08L95.6 76.02L106.13 69.48L116.67 85.91L127.2 77.72L137.73 77.3L148.27 77.51L158.8 85.11L169.33 73.96L179.87 84.58L190.4 80.21L200.93 82.62L211.47 79.93L222 81.88L232.53 79.75L243.07 77L253.6 79.42L264.13 81.59L274.67 81.72L285.2 81.23L295.73 79.08L306.27 83.63L316.8 81.09L327.33 73.16L337.87 81.57L348.4 83.5L358.93 80.75L369.47 82.58L380 72.79L390.53 70L401.07 79.08L411.6 69.4L422.13 67.14L432.67 74.45L443.2 75.7L453.73 72.11L464.27 191.53L474.8 75.57L485.33 73.64L495.87 74.93L506.4 78.08L516.93 75.32L527.47 70.33L538 70.74L548.53 76.09L559.07 78.35L569.6 77.96L580.13 69.18L590.67 78.82L601.2 78.76L611.73 75.33L622.27 78.65L632.8 69.95L643.33 75.34' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> PR · x=0: 0.775 PR · x=1: 0.804 PR · x=2: 0.781 PR · x=3: 0.796 PR · x=4: 0.82 PR · x=5: 0.76 PR · x=6: 0.79 PR · x=7: 0.791 PR · x=8: 0.791 PR · x=9: 0.763 PR · x=10: 0.804 PR · x=11: 0.765 PR · x=12: 0.781 PR · x=13: 0.772 PR · x=14: 0.782 PR · x=15: 0.775 PR · x=16: 0.782 PR · x=17: 0.793 PR · x=18: 0.784 PR · x=19: 0.776 PR · x=20: 0.775 PR · x=21: 0.777 PR · x=22: 0.785 PR · x=23: 0.768 PR · x=24: 0.777 PR · x=25: 0.807 PR · x=26: 0.776 PR · x=27: 0.769 PR · x=28: 0.779 PR · x=29: 0.772 PR · x=30: 0.808 PR · x=31: 0.819 PR · x=32: 0.785 PR · x=33: 0.821 PR · x=34: 0.829 PR · x=35: 0.802 PR · x=36: 0.797 PR · x=37: 0.811 PR · x=38: 0.368 PR · x=39: 0.798 PR · x=40: 0.805 PR · x=41: 0.8 PR · x=42: 0.789 PR · x=43: 0.799 PR · x=44: 0.817 PR · x=45: 0.816 PR · x=46: 0.796 PR · x=47: 0.788 PR · x=48: 0.789 PR · x=49: 0.822 PR · x=50: 0.786 PR · x=51: 0.786 PR · x=52: 0.799 PR · x=53: 0.786 PR · x=54: 0.819 PR · x=55: 0.799

## 2 · Prevedere la potenza di domani, sapendo il meteo previsto
La potenza dipende dall'irraggiamento e un servizio meteo lo prevede il giorno prima: è la **covariata nota nel futuro** (con il suo errore). Modello: regressione $P\approx\beta_0+\beta_1G_{prev}+\beta_2G_{prev}\,T_{aria}$, limitata a 0…600 kW, addestrata a QR sulle ore in cui l'inverter era in servizio. Riferimento: il **seasonal naive a un giorno** (la stessa ora di ieri).

In [ ]:
var trainRows = Enumerable.Range(0, trainHours).Where(i => ok[i] == 1).ToArray();   // fuori le ore di guasto: non sono "normali"
var X = new double[trainRows.Length, 3];
var y = new double[trainRows.Length];
for (var r = 0; r < trainRows.Length; r++)
{
    var i = trainRows[r];
    X[r, 0] = 1; X[r, 1] = Gf[i]; X[r, 2] = Gf[i] * Tair[i];
    y[r] = Pac[i];
}
var beta = MathemaSharp.LinearAlgebra.MatrixAlgebra.LeastSquares<double>(X, y);
P("beta", beta);
double Predict(int i) => Gf[i] <= 0 ? 0 : Math.Clamp(beta[0] + beta[1] * Gf[i] + beta[2] * Gf[i] * Tair[i], 0, pMax);

double[] train = Pac[..trainHours], test = Pac[trainHours..];
var model = Enumerable.Range(trainHours, 168).Select(Predict).ToArray();
var persistence = Pac[(trainHours - 24)..(total - 24)];                              // ieri alla stessa ora
P("mase_modello", Accuracy.Mase<double>(test, model, train, 24));
P("mase_persistenza", Accuracy.Mase<double>(test, persistence, train, 24));
try { Accuracy.Mape<double>(test, model); P("mape_definita", 1); } catch (ArgumentException) { P("mape_definita", 0); }   // di notte la potenza è 0

beta: 10.992833 0.806969 -0.002707
mase_modello: 0.510989
mase_persistenza: 1.361532
mape_definita: 0.000000

**La MAPE non si può calcolare**: di notte la potenza vale zero e la libreria lo dichiara invece di restituire un numero senza senso. È il motivo per cui in questo libro si usa la **MASE**: definita anche con gli zeri e indipendente dalla scala.

## 3 · Con quale incertezza? Quantili dei residui per ora del giorno
Le previsioni sbagliano in modo diverso a ogni ora (a mezzogiorno l'errore è grande, di notte nullo). Si prendono i **residui in campione** del modello (potenza reale − prevista) **ora per ora** e se ne calcolano i quantili: sono gli scarti da sommare alla previsione per ottenere P10, P50, P90.

In [ ]:
var residual = new double[trainHours];
for (var i = 0; i < trainHours; i++) residual[i] = Pac[i] - Predict(i);
double[] HourQuantile(int hour, double p) => new[] { SeriesStatistics.Quantile<double>(trainRows.Where(i => i % 24 == hour).Select(i => residual[i]).ToArray(), p) };
var q10 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.1)[0]).ToArray();
var q50 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.5)[0]).ToArray();
var q90 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.9)[0]).ToArray();

double Band(int i, double[] q) => Gf[i] <= 0 ? 0 : Math.Clamp(Predict(i) + q[i % 24], 0, pMax);
var lo = Enumerable.Range(trainHours, 168).Select(i => Band(i, q10)).ToArray();
var mid = Enumerable.Range(trainHours, 168).Select(i => Band(i, q50)).ToArray();
var hi = Enumerable.Range(trainHours, 168).Select(i => Band(i, q90)).ToArray();
P("pinball_p10", Accuracy.PinballLoss<double>(test, lo, 0.1));
P("pinball_p50", Accuracy.PinballLoss<double>(test, mid, 0.5));
P("pinball_p90", Accuracy.PinballLoss<double>(test, hi, 0.9));
P("copertura_p10_p90_tutte_le_ore", Accuracy.IntervalCoverage<double>(test, lo, hi));
// Di notte la potenza è 0 e l'intervallo è [0, 0]: contiene sempre il valore e gonfia la copertura. Si misura solo sulle ore di sole.
var daylight = Enumerable.Range(0, 168).Where(t => Gf[trainHours + t] > 0).ToArray();
double[] At(double[] x) => daylight.Select(t => x[t]).ToArray();
var coverageDay = Accuracy.IntervalCoverage<double>(At(test), At(lo), At(hi));
P("copertura_p10_p90_ore_di_sole", coverageDay);

pinball_p10: 4.859385
pinball_p50: 11.591131
pinball_p90: 4.671934
copertura_p10_p90_tutte_le_ore: 0.916667
copertura_p10_p90_ore_di_sole: 0.846154

L'intervallo P10–P90 dovrebbe contenere l'**80 %** delle osservazioni. Contando **tutte** le ore la copertura sembra alta, ma è un'illusione: di notte la potenza è zero e l'intervallo è [0, 0], quindi contiene sempre il valore. Misurata **solo sulle ore di sole** è il numero che conta; se risultasse più bassa dell'80 %, l'intervallo sarebbe troppo ottimista (i residui in campione sottostimano l'errore su dati nuovi).

In [ ]:
int a = 96, b = 168;   // gli ultimi 3 giorni della prova
Chart.Lines("Ultimi 3 giorni: potenza reale e fascia P10–P90 (kW, asse x: ore)",
    new Line("reale", test[a..b]), new Line("P50", mid[a..b], Dashed: true), new Line("P10", lo[a..b], Dashed: true), new Line("P90", hi[a..b], Dashed: true));

Ultimi 3 giorni: potenza reale e fascia P10–P90 (kW, asse x: ore) 0 200 400 600 0 20 40 60 80 <path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 208.66L127.2 178.7L135.1 148.54L143 137.62L150.9 124.06L158.8 95.11L166.7 83.02L174.6 89.5L182.5 102.77L190.4 126.6L198.3 150.46L206.2 182.93L214.1 208.85L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 209.62L316.8 192.67L324.7 189.67L332.6 188.4L340.5 172.99L348.4 183.96L356.3 175.78L364.2 187.49L372.1 170.18L380 184.42L387.9 182.33L395.8 196.68L403.7 209.69L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 209.83L506.4 183.67L514.3 159.14L522.2 133.34L530.1 132.19L538 125.54L545.9 115.13L553.8 117.62L561.7 138.12L569.6 136.06L577.5 161.59L585.4 187.66L593.3 209.83L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 207.18L127.2 181.22L135.1 150.26L143 125.13L150.9 102.97L158.8 108.35L166.7 101.37L174.6 98.02L182.5 111.62L190.4 124.45L198.3 152.52L206.2 179.32L214.1 206.92L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 207.75L316.8 186.74L324.7 166.38L332.6 159.55L340.5 147.96L348.4 154.35L356.3 155.6L364.2 154.59L372.1 159.31L380 170.92L387.9 179.59L395.8 193.14L403.7 207.73L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 208.06L506.4 183.25L514.3 155.27L522.2 131.26L530.1 118.96L538 125.87L545.9 124.06L553.8 118.03L561.7 122.48L569.6 131.97L577.5 162.69L585.4 186.97L593.3 207.91L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#eb6834' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 210L127.2 191.07L135.1 168.45L143 152.56L150.9 132.68L158.8 135.24L166.7 123.8L174.6 118.61L182.5 133.15L190.4 149.29L198.3 165.08L206.2 187.61L214.1 209.95L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 210L316.8 196.6L324.7 184.58L332.6 186.97L340.5 177.66L348.4 181.25L356.3 178.02L364.2 175.18L372.1 180.84L380 195.76L387.9 192.15L395.8 201.43L403.7 210L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 210L506.4 193.11L514.3 173.47L522.2 158.68L530.1 148.66L538 152.76L545.9 146.48L553.8 138.62L561.7 144.02L569.6 156.81L577.5 175.25L585.4 195.27L593.3 210L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#1baf7a' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 200.64L127.2 171.32L135.1 135.25L143 104.52L150.9 77.43L158.8 80.98L166.7 78.86L174.6 74L182.5 89.58L190.4 105.79L198.3 134.04L206.2 170.48L214.1 202.01L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 201.2L316.8 176.85L324.7 151.37L332.6 138.93L340.5 122.41L348.4 126.98L356.3 133.08L364.2 130.57L372.1 137.27L380 152.27L387.9 161.1L395.8 184.3L403.7 202.82L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 201.52L506.4 173.35L514.3 140.27L522.2 110.64L530.1 93.41L538 98.49L545.9 101.54L553.8 94.01L561.7 100.44L569.6 113.31L577.5 144.21L585.4 178.13L593.3 203L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#eda100' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/> reale · x=0: 0 reale · x=1: 0 reale · x=2: 0 reale · x=3: 0 reale · x=4: 0 reale · x=5: 0 reale · x=6: 0 reale · x=7: 5.6 reale · x=8: 130 reale · x=9: 256 reale · x=10: 302 reale · x=11: 358 rea

## 4 · E l'energia dell'**intera giornata**?
La domanda del trader riguarda i kWh del giorno, non la potenza di un'ora. **Sommare i P10 orari non dà il P10 del giorno**: equivale a supporre che tutte le 24 ore sbaglino insieme nel peggiore dei modi. Si preservano invece le **dipendenze nel tempo** campionando traiettorie: a ogni giornata di addestramento si prende il suo intero vettore di residui, lo si somma alla previsione e si ottiene un'energia possibile. Dalle 48 energie possibili si prendono i quantili (giorno del guasto escluso).

In [ ]:
var okDays = Enumerable.Range(0, trainDays).Where(d => d != 38).ToArray();
var p10Day = new double[7]; var p50Day = new double[7]; var p90Day = new double[7]; var p25Day = new double[7];
var actualDay = new double[7]; var sumLo = new double[7]; var sumHi = new double[7];
var trajectoryTotals = new double[7][];
for (var d = 0; d < 7; d++)
{
    var totals = new double[okDays.Length];
    for (var k = 0; k < okDays.Length; k++)
    {
        double s = 0;
        for (var h = 0; h < 24; h++)
        {
            var i = trainHours + d * 24 + h;
            s += Gf[i] <= 0 ? 0 : Math.Clamp(model[d * 24 + h] + residual[okDays[k] * 24 + h], 0, pMax);
        }
        totals[k] = s;
    }
    trajectoryTotals[d] = totals;
    p10Day[d] = SeriesStatistics.Quantile<double>(totals, 0.1);
    p25Day[d] = SeriesStatistics.Quantile<double>(totals, 0.25);
    p50Day[d] = SeriesStatistics.Quantile<double>(totals, 0.5);
    p90Day[d] = SeriesStatistics.Quantile<double>(totals, 0.9);
    for (var h = 0; h < 24; h++) { actualDay[d] += test[d * 24 + h]; sumLo[d] += lo[d * 24 + h]; sumHi[d] += hi[d * 24 + h]; }
}
P("energia_reale", actualDay);
P("energia_p10", p10Day);
P("energia_p50", p50Day);
P("energia_p90", p90Day);
P("energia_somma_p10", sumLo);
P("energia_somma_p90", sumHi);
int Inside(double[] low, double[] high) => Enumerable.Range(0, 7).Count(d => actualDay[d] >= low[d] && actualDay[d] <= high[d]);
P("giorni_dentro_traiettorie", Inside(p10Day, p90Day));
P("giorni_dentro_somma_oraria", Inside(sumLo, sumHi));
P("larghezza_traiettorie_kwh", Enumerable.Range(0, 7).Average(d => p90Day[d] - p10Day[d]));
P("larghezza_somma_oraria_kwh", Enumerable.Range(0, 7).Average(d => sumHi[d] - sumLo[d]));

energia_reale: 1414.800000 4036.600000 3908.600000 3150.400000 3721.700000 1192.200000 3001.100000
energia_p10: 914.216513 2432.517031 2803.159339 1480.914654 2835.150288 1195.885183 2308.782712
energia_p50: 1748.521896 3269.379526 3639.933614 2315.279025 3670.481569 2027.662215 3139.423999
energia_p90: 2443.001177 3963.858806 4334.412895 3009.758305 4364.960850 2722.141495 3833.903279
energia_somma_p10: 796.324391 2315.913303 2686.467392 1365.298881 2718.824564 1081.509182 2195.316611
energia_somma_p90: 2682.711908 4203.569537 4574.123626 3249.469036 4604.671581 2961.852226 4073.614010
giorni_dentro_traiettorie: 4.000000
giorni_dentro_somma_oraria: 7.000000
larghezza_traiettorie_kwh: 1528.773012
larghezza_somma_oraria_kwh: 1884.336800

La banda **dalle traiettorie** è più stretta di quella ottenuta sommando i quantili orari (circa **1.5 MWh** contro **1.9 MWh** in media): tiene conto di quanto gli errori si compensano durante il giorno, mentre la somma assume il caso peggiore su tutte le ore insieme, ed è per questo che contiene sempre tutti e sette i giorni. I giorni dentro la banda delle traiettorie sono **4 su 7** contro l'80 % atteso: con soli 7 giorni è compatibile con la sorte (vedere 4 o meno su 7 con probabilità 0.8 capita circa 15 volte su 100). **Non è una calibrazione**: ci vogliono molte più origini di prova.

## 5 · Quanta energia offrire? Il quantile che costa meno
Sul mercato, **produrre meno di quanto offerto costa di più** che produrre di più: se lo scarto per difetto costa $a$ per kWh e quello per eccesso $b$, l'offerta che minimizza il costo atteso è il **quantile di livello $b/(a+b)$** della distribuzione dell'energia: è la perdita pinball del capitolo sulle previsioni probabilistiche. Con $a=3,\;b=1$ il livello è **0.25**.

In [ ]:
double Cost(double offer, double actual) => 3 * Math.Max(offer - actual, 0) + 1 * Math.Max(actual - offer, 0);   // a = 3 (difetto), b = 1 (eccesso)
double WeekCost(double[] offers) => Enumerable.Range(0, 7).Sum(d => Cost(offers[d], actualDay[d]));
P("costo_offerta_p10", WeekCost(p10Day));
P("costo_offerta_p25", WeekCost(p25Day));
P("costo_offerta_p50", WeekCost(p50Day));
P("costo_offerta_p90", WeekCost(p90Day));
var bestOffer = new[] { ("p10", p10Day), ("p25", p25Day), ("p50", p50Day), ("p90", p90Day) }.OrderBy(o => WeekCost(o.Item2)).First();
Console.WriteLine($"offerta_migliore: {bestOffer.Item1}");

costo_offerta_p10: 6469.515011
costo_offerta_p25: 5093.702594
costo_offerta_p50: 5844.750598
costo_offerta_p90: 13593.441976
offerta_migliore: p25

Sulla settimana di prova il livello 0.25 costa meno di offrire la mediana (P50) e molto meno del P90, come la teoria dice. Con sette giorni è una conferma, non una dimostrazione: la teoria vale **in media**, e il costo di una settimana è rumoroso.

## 6 · Comunicare: KPI, dashboard e risposta al cliente

In [ ]:
var lastDay = 6;
Kpi.Tiles(
    new Tile("Performance ratio", $"{MeanPr(31, 37):F3}", $"dopo la pulizia, contro {MeanPr(24, 30):F3} prima", "ok"),
    new Tile("Disponibilità", $"{100.0 * sunny.Count(i => ok[i] == 1) / sunny.Length:F1} %", "ore di sole con inverter in servizio; 5 ore di guasto il giorno 38", "ok"),
    new Tile("Energia domani (P25)", $"{p25Day[lastDay]:F0} kWh", $"P10 {p10Day[lastDay]:F0} – P90 {p90Day[lastDay]:F0} kWh; mediana {p50Day[lastDay]:F0}", "ok"),
    new Tile("Copertura P10–P90", $"{100 * coverageDay:F0} %", "ore di sole, sulla settimana di prova (atteso 80 %)", coverageDay < 0.7 ? "warn" : "ok"),
    new Tile("MASE del modello", $"{Accuracy.Mase<double>(test, model, train, 24):F2}", $"contro {Accuracy.Mase<double>(test, persistence, train, 24):F2} della persistenza (ieri = oggi)", "ok"));

Performance ratio 0.809 dopo la pulizia, contro 0.780 prima ✓ nella norma Disponibilità 99.3 % ore di sole con inverter in servizio; 5 ore di guasto il giorno 38 ✓ nella norma Energia domani (P25) 2768 kWh P10 2309 – P90 3834 kWh; mediana 3139 ✓ nella norma Copertura P10–P90 85 % ore di sole, sulla settimana di prova (atteso 80 %) ✓ nella norma MASE del modello 0.51 contro 1.36 della persistenza (ieri = oggi) ✓ nella norma

In [ ]:
var answer = $"Per il 20 settembre offriamo {p25Day[lastDay]:F0} kWh. È il livello 25 % della distribuzione dell'energia attesa (mediana {p50Day[lastDay]:F0} kWh, intervallo 10–90 % da {p10Day[lastDay]:F0} a {p90Day[lastDay]:F0} kWh): " +
             $"il mercato penalizza il difetto tre volte l'eccesso, e offrire il livello {100 * 1.0 / 4:F0} % è ciò che minimizza il costo atteso. " +
             $"La previsione si basa sull'irraggiamento previsto dal servizio meteo: se il meteo cambia, aggiorneremo l'offerta entro la chiusura del mercato.";
Kpi.Quote("Risposta al cliente: «quanta energia offrite sul mercato domani?»", answer);

var oem = $"Il performance ratio medio è salito da {MeanPr(24, 30):F3} a {MeanPr(31, 37):F3} dopo la pulizia dei moduli del giorno 30: lo sporco costava circa il {100 * (MeanPr(31, 37) - MeanPr(24, 30)) / MeanPr(31, 37):F1} % di energia. Il giorno del guasto (38) il PR è sceso a {pr[38]:F2}: 5 ore di inverter fermo.";
Kpi.Quote("Nota per O&M", oem);

Risposta al cliente: «quanta energia offrite sul mercato domani?» Per il 20 settembre offriamo 2768 kWh. È il livello 25 % della distribuzione dell'energia attesa (mediana 3139 kWh, intervallo 10–90 % da 2309 a 3834 kWh): il mercato penalizza il difetto tre volte l'eccesso, e offrire il livello 25 % è ciò che minimizza il costo atteso. La previsione si basa sull'irraggiamento previsto dal servizio meteo: se il meteo cambia, aggiorneremo l'offerta entro la chiusura del mercato.

Nota per O&M Il performance ratio medio è salito da 0.780 a 0.809 dopo la pulizia dei moduli del giorno 30: lo sporco costava circa il 3.6 % di energia. Il giorno del guasto (38) il PR è sceso a 0.37: 5 ore di inverter fermo.

## Cosa abbiamo imparato
- **Il KPI giusto toglie il meteo**: il PR separa la salute dell'impianto dal tempo che fa.
- **Una covariata nota migliora molto** la previsione, ma porta con sé il suo errore: l'incertezza della previsione meteo *è* l'incertezza dell'offerta.
- **MAPE no, MASE sì**: con gli zeri notturni la MAPE non esiste.
- **I quantili orari non si sommano**: per l'energia del giorno si campionano traiettorie, preservando le dipendenze nel tempo.
- **Il costo dell'errore decide il quantile**: offrire la mediana non è ottimo quando il difetto costa più dell'eccesso.
- **Una settimana di prova non basta** per calibrare l'intervallo: la validazione su più origini (*rolling origin*) e la previsione conforme sono nel libro, non ancora nella libreria.